In [4]:
import os, sys
sys.path.append(os.path.abspath('../..'))
from utlis.sync_utlis.general_loader import load_flat_with_frame_map, merge_pred_with_miniscope
from utlis.sync_utlis.general_loader_viz import plot_two_coms_from_pred_df


oct3v1 = "/data/big_rim/rsync_dcc_sum/Oct3V1"
rec_path = os.path.join(oct3v1, "2024_12_18/20240919v1l5r1mini_p20240717PMC_social_test_11_30")
nc_key= "wnd1000_stp700_max25_diff3.5_pnrauto"
# 'wnd1500_stp700_max25_diff3.5_pnr1.1'
dannce_folder='SDANNCE/predict00'


merged = merge_pred_with_miniscope(
    rec_path=rec_path,
    nc_key=nc_key,
    dannce_folder=dannce_folder,
    com_folder=None,
    save_h5=False,
    save_csv=False,
)

merged.shape, merged.index[:3]

((12700, 237), Index([-5, 35, 64], dtype='int64', name='timestamp_ms_mini'))

In [5]:
merged

,com1_x,com2_x,com1_y,com2_y,com1_z,com2_z,kp1_x_a1,kp2_x_a1,kp3_x_a1,kp4_x_a1,...,dF_F_roi39,dF_F_roi40,dF_F_roi41,dF_F_roi42,dF_F_roi43,dF_F_roi44,dF_F_roi45,dF_F_roi46,dF_F_roi47,dF_F_roi48
timestamp_ms_mini,,,,,,,,,,,,,,,,,,,,,
-5,-412.380270,-372.665627,168.820058,200.239392,5.079139,7.018563,-431.659027,-402.266022,-407.163361,-415.559296,...,-0.487672,0.764167,0.703535,2.283586,0.455371,-0.217957,1.073101,0.270197,1.832038,0.955422
35,-412.167869,-372.665627,168.820058,200.239392,5.129800,6.895860,-431.641388,-402.121399,-406.700989,-415.339935,...,0.159921,0.666013,0.871495,2.528362,-0.222600,-0.493738,0.230324,-0.340074,1.174880,0.445190
64,-412.167869,-372.661515,168.739270,200.244981,5.138774,6.874564,-431.271637,-401.848358,-406.995605,-415.006165,...,-0.329742,1.070483,0.725788,2.227845,0.245084,-0.090056,0.273174,-0.505719,1.091894,1.001543
97,-412.167869,-372.567755,168.112491,200.263531,5.164036,6.874564,-430.337097,-401.019714,-408.643646,-414.297760,...,-0.356897,0.534905,0.283326,2.244527,0.367181,-0.071263,0.741474,0.301717,1.988927,1.271346
130,-412.167869,-372.346276,168.062731,200.263531,5.168815,6.865812,-430.316772,-400.528381,-411.645172,-414.481903,...,-0.179207,0.365770,0.638216,2.507418,0.717327,-0.057563,0.376802,-0.233521,2.081703,1.212617
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
426643,-74.527784,-369.190341,297.357226,324.684113,10.250276,2.164954,-57.335758,-87.638435,-62.119793,-72.914978,...,0.452275,1.594847,0.459956,0.943885,0.609586,1.241530,5.182661,0.845629,1.594539,0.464482
426677,-74.527784,-369.190341,293.014935,324.684113,10.250276,2.101645,-54.560120,-86.277748,-61.356266,-71.028160,...,0.052685,2.033458,-0.104122,0.074269,0.642538,0.258096,3.862572,0.705147,1.841283,0.768249
426710,-74.552294,-369.190341,288.680901,324.684113,10.250276,2.011307,-54.927963,-85.768623,-61.791847,-70.873260,...,-0.335192,2.257697,-0.350771,0.873994,0.492468,0.339826,3.881438,0.911479,0.848227,0.283027


In [6]:
from utlis.social_analysis.approaching import compute_com_distance, compute_motion_direction, detect_approaches
# 0) If you want distance only
dist = compute_com_distance(merged, p1="com1", p2="com2", smooth_window=5)
print(dist.head())

# 1) Motion vectors / directions for each animal
m1 = compute_motion_direction(merged, prefix="com1", time_col="timestamp_ms_mini", pos_smooth=5, vel_smooth=5)
m2 = compute_motion_direction(merged, prefix="com2", time_col="timestamp_ms_mini", pos_smooth=5, vel_smooth=5)
print(m1.filter(like="_speed").head(), m2.filter(like="_speed").head())

# 2) Detect approach windows (tune thresholds as you like)
res = detect_approaches(
    merged,
    p1="com1", p2="com2",
    time_col="timestamp_ms_mini",   # if absent, pass fps=...
    pos_smooth=5, vel_smooth=5,
    radial_thresh=20.0,             # mm/s toward the other
    speed_min=5.0,                  # mm/s total speed floor
    dist_min=None,                  # e.g., set 50.0 to ignore near-collisions if needed
    dist_max=300.0,                 # focus on interaction zone
    min_samples=15,                 # ≥15 consecutive frames
    return_intervals=True
)

# Per-frame booleans + metrics
res["frames"][["dist_mm","radial1","radial2","approach1","approach2","mutual"]].head()

# Interval summaries (list of dicts)
res["intervals"]["approach1"][:3], res["intervals"]["approach2"][:3], res["intervals"]["mutual"][:3]


timestamp_ms_mini
-5      50.580713
 35     50.694257
 64     50.803232
 97     50.946379
 130    51.173287
Name: dist_mm, dtype: float64
                   com1_speed
timestamp_ms_mini            
-5                   0.154367
 35                  0.162126
 64                  0.167439
 97                  0.159762
 130                 0.147180                    com2_speed
timestamp_ms_mini            
-5                   0.047668
 35                  0.057717
 64                  0.065080
 97                  0.077614
 130                 0.081325


([], [], [])

In [7]:
def pick_alignment_cols(frames):
    frame_col = None
    for c in ("camera_frame_sixcam", "mapped_sixcam_frame_indices"):
        if c in frames.columns:
            frame_col = c
            break
    mini_ms_col = "timestamp_ms_mini" if "timestamp_ms_mini" in frames.columns else None
    return frame_col, mini_ms_col


def extract_incidents(frames, events, frame_col=None, mini_ms_col=None):
    """
    Turns events into simple per-event lists you can feed to other funcs.
    Returns a list of dicts:
      {"start_idx", "end_idx_exclusive", "index", "sixcam", "mini_ms"}
    Missing cols are omitted.
    """
    out = []
    for ev in events:
        s, e = ev["start_idx"], ev["end_idx_exclusive"]
        sl = slice(s, e)
        item = {
            "start_idx": s,
            "end_idx_exclusive": e,
            "index": frames.index[sl].tolist()
        }
        if frame_col:
            item["sixcam"] = frames.iloc[sl][frame_col].astype(int).tolist()
        if mini_ms_col:
            item["mini_ms"] = frames.iloc[sl][mini_ms_col].astype(float).tolist()
        out.append(item)
    return out


import numpy as np
import matplotlib.pyplot as plt

def plot_dist_with_events(frames, mask, events, contact_mm=50.0, title=None):
    dist = frames["dist_mm"].to_numpy()
    x = frames.index

    fig, ax = plt.subplots(figsize=(10, 3.5))
    ax.plot(x, dist, lw=1, label="dist_mm")
    ax.scatter(x[mask], dist[mask], s=8, label="approach_success", zorder=3)
    ax.axhline(contact_mm, ls="--", lw=1, label=f"contact={contact_mm} mm")

    if events:
        contact_idx = np.array([ev["contact_idx"] for ev in events], dtype=int)
        ax.scatter(x[contact_idx], dist[contact_idx], marker="x", s=36, zorder=4, label="first contact")

        # light spans per event (fast; number of events is small)
        for ev in events:
            s, e = ev["start_idx"], ev["end_idx_exclusive"]
            ax.axvspan(x[s], x[e-1], alpha=0.15)

    ax.set_ylabel("distance (mm)")
    ax.set_xlabel("frame")  # or "time" if your index is time-like
    ax.set_title(title or f"Approach-success events (n={len(events)})")
    ax.legend(loc="best")
    plt.tight_layout()
    return fig, ax


# def plot_single_event(frames, ev, contact_mm=50.0):
#     s, e = ev["start_idx"], ev["end_idx_exclusive"]
#     sl = slice(s, e)
#     x = frames.index[sl]
#     dist = frames["dist_mm"].iloc[sl]

#     fig, ax = plt.subplots(figsize=(8, 3))
#     ax.plot(x, dist, lw=1.2)
#     ax.axhline(contact_mm, ls="--", lw=1)
#     # mark first contact if ev has it
#     if "contact_idx" in ev:
#         cx = frames.index[ev["contact_idx"]]
#         ax.scatter([cx], [frames["dist_mm"].iloc[ev["contact_idx"]]], marker="x", s=36, zorder=4)
#     ax.set_ylabel("distance (mm)")
#     ax.set_xlabel("frame")
#     ax.set_title(f"Event [{s}:{e})  drop={ev.get('drop_mm', np.nan):.1f}mm")
#     plt.tight_layout()
#     return fig, ax

def plot_single_event(frames, ev, contact_mm=50.0, mark="both"):
    """
    mark: "contact" | "bottom" | "both"
    """
    s, e = ev["start_idx"], ev["end_idx_exclusive"]
    x = frames.index[s:e]
    dist = frames["dist_mm"].iloc[s:e]

    fig, ax = plt.subplots(figsize=(8, 3))
    ax.plot(x, dist, lw=1.2, zorder=1)
    ax.axhline(contact_mm, ls="--", lw=1, zorder=0)

    # start/end verticals for context
    ax.axvline(frames.index[s], ls=":", lw=1, alpha=0.6, zorder=0)
    ax.axvline(frames.index[e-1], ls=":", lw=1, alpha=0.6, zorder=0)

    if ("contact_idx" in ev) and (mark in {"contact", "both"}):
        ci = ev["contact_idx"]
        ax.scatter([frames.index[ci]], [frames["dist_mm"].iat[ci]],
                   marker="x", s=40, zorder=3, label="first contact")

    if ("bottom_idx" in ev) and (mark in {"bottom", "both"}):
        bi = ev["bottom_idx"]
        ax.scatter([frames.index[bi]], [frames["dist_mm"].iat[bi]],
                   marker="o", s=36, zorder=4, label="bottom (min distance)")

    ax.set_ylabel("distance (mm)")
    ax.set_xlabel("frame")
    ax.set_title(f"Event [{s}:{e})  drop={ev.get('drop_mm', float('nan')):.1f} mm")
    ax.legend(loc="best", frameon=False)
    fig.tight_layout()
    return fig, ax



In [8]:
from utlis.social_analysis.approaching import _boolean_runs, find_approach_success


contact_mm = 50
# --- choose a setting (use your "best" from the sweep) ---
mask, events = find_approach_success(
    res["frames"],
    contact_mm=contact_mm,
    dD_dt_thresh=0.0,
    min_len=1,
    min_drop_mm=1#10.0
)

# mask0, events0 = find_approach_success(res["frames"], contact_mm=50, dD_dt_thresh=0.0, min_len=1, min_drop_mm=1)
# mask, events = merge_overlapping_approach_events(res["frames"], events0, contact_mm=150, max_gap=0, min_len=1, min_drop_mm=1)

frames = res["frames"]

# --- global plot with event spans (uses the helper we wrote) ---
# plot_dist_with_events(frames, mask, events, contact_mm=contact_mm)

# --- quick QA: first 3 events as small windows ---
# for ev in events[:3]:
#     plot_single_event(frames, ev, contact_mm=contact_mm)

In [9]:
# from utlis.social_analysis.approaching import find_approach_success
from utlis.vis_valid_utlis.sdannce_vis import cfg, visualize_frames

C = cfg(base_path="/data/big_rim/rsync_dcc_sum/Oct3V1/2024_12_18/20240919v1l5r1mini_p20240717PMC_social_test_11_30", cammm=1, 
        enable_zoom=True, zoom_margin=450, video_fps=30) #animal="mouse20", #write_pngs=True

# 0) Run your detector first
mask, events = find_approach_success(
    res["frames"],
    contact_mm=contact_mm,
    dD_dt_thresh=0.0,
    min_len=1,
    min_drop_mm=1.0
)

frames = res["frames"]
frame_col = "camera_frame_sixcam" if "camera_frame_sixcam" in frames.columns \
    else ("mapped_sixcam_frame_indices" if "mapped_sixcam_frame_indices" in frames.columns else None)

print(f"#approach_success events: {len(events)}")

incident_all_six_cam = frames.loc[mask, frame_col].astype(int).tolist() if frame_col else frames.index[mask].tolist()

# visualize_frames(incident_all_six_cam, config=C, out_name=f"251005_specifc_frame_all_incidents_contact{contact_mm}_dD0_min1d1")

#approach_success events: 4


In [10]:
len(incident_all_six_cam)

254

# miniscope overlay video below

In [11]:
import os, json, subprocess, tempfile, shutil
from pathlib import Path
import os, json, subprocess, tempfile
from pathlib import Path
import pandas as pd


# --- 1) Exact lookup with Buffer Index included ---
def map_exact_timestamps_to_frames_with_buffer(csv_path, timestamps_ms):
    """
    Exact lookup. Returns DataFrame:
      timestamp_ms, frame_number (global), buffer_idx
    Raises if any timestamp is missing.
    """
    df = pd.read_csv(csv_path, usecols=["Frame Number", "Time Stamp (ms)", "Buffer Index"])
    q = pd.DataFrame({"Time Stamp (ms)": list(timestamps_ms)})
    out = q.merge(df, on="Time Stamp (ms)", how="left").rename(
        columns={
            "Time Stamp (ms)": "timestamp_ms",
            "Frame Number": "frame_number",
            "Buffer Index": "buffer_idx",
        }
    )
    if out["frame_number"].isna().any() or out["buffer_idx"].isna().any():
        missing = out.loc[out["frame_number"].isna() | out["buffer_idx"].isna(), "timestamp_ms"].tolist()
        raise ValueError(f"Timestamps not found in CSV: {missing}")
    out["frame_number"] = out["frame_number"].astype(int)
    out["buffer_idx"] = out["buffer_idx"].astype(int)
    return out[["timestamp_ms", "frame_number", "buffer_idx"]]


def load_miniscope_meta(video_dir):
    meta = json.load(open(Path(video_dir) / "metaData.json", "r"))
    return {
        "frameRate": float(meta.get("frameRate", 30)),
        "framesPerFile": int(meta.get("framesPerFile", 1000)),
    }

def _ffprobe_nb_frames(path: Path) -> int:
    """
    Return the number of frames ffprobe sees (as int). Raises if file can't be probed.
    """
    cmd = [
        "ffprobe", "-v", "error", "-count_frames", "-select_streams", "v:0",
        "-show_entries", "stream=nb_read_frames", "-of", "default=nokey=1:noprint_wrappers=1",
        str(path)
    ]
    cp = subprocess.run(cmd, capture_output=True, text=True)
    if cp.returncode != 0:
        raise RuntimeError(f"ffprobe failed on {path}:\n{cp.stderr}")
    s = cp.stdout.strip()
    try:
        return int(s)
    except Exception:
        # Sometimes nb_read_frames can be "N/A" for some codecs; fall back to pkt-based probe
        cmd2 = [
            "ffprobe", "-v", "error", "-select_streams", "v:0",
            "-show_entries", "stream=nb_frames", "-of", "default=nokey=1:noprint_wrappers=1",
            str(path)
        ]
        cp2 = subprocess.run(cmd2, capture_output=True, text=True)
        if cp2.returncode != 0:
            raise RuntimeError(f"ffprobe (nb_frames) failed on {path}:\n{cp2.stderr}")
        s2 = cp2.stdout.strip()
        return int(s2) if s2.isdigit() else 0

def _build_select_eq(local_indices):
    """
    Build a comma-free select string: select='eq(n,idx1)+eq(n,idx2)+...'
    Avoids all comma-escaping pitfalls with subprocess.
    """
    terms = [f"eq(n,{int(i)})" for i in sorted(set(local_indices))]
    expr = "+".join(terms) if terms else "0"  # '0' -> selects nothing
    return f"select='{expr}'"

def write_frames_video_by_fileidx_ffmpeg_fast(
    video_dir, selections_df, out_path, fps=None, preset="ultrafast"
):
    """
    Fast FFmpeg path, robust for FFV1.
    - Derives file index from global frame number and framesPerFile.
    - Validates indices against actual nb_frames per file (drops OOR gracefully).
    - Uses select='eq(n,...) + eq(n,...)' to avoid comma escaping.
    """
    video_dir = Path(video_dir)
    os.makedirs(Path(out_path).parent, exist_ok=True)

    meta = load_miniscope_meta(video_dir)
    fps_use = float(meta["frameRate"] if fps is None else fps)
    fpf = int(meta["framesPerFile"])

    S = selections_df.copy()
    S["file_idx"] = (S["frame_number"] // fpf).astype(int)
    S["local"]    = (S["frame_number"] %  fpf).astype(int)
    S = S.sort_values("timestamp_ms").reset_index(drop=True)

    tmpdir = Path(tempfile.mkdtemp(prefix="mini_fast_"))
    extracted = {}  # (file_idx, local) -> path
    try:
        # 1) Batch-extract per file
        for fi, group in S.groupby("file_idx"):
            fi = int(fi)
            src = video_dir / f"{fi}.avi"
            if not src.exists():
                raise FileNotFoundError(f"Missing split file: {src}")

            # Probe actual frame count and clip requested locals to valid range
            try:
                nb = _ffprobe_nb_frames(src)
            except Exception as e:
                raise RuntimeError(f"ffprobe failed on {src}: {e}")

            locals_req = sorted(set(int(x) for x in group["local"].tolist()))
            locals_ok  = [x for x in locals_req if 0 <= x < nb]
            if not locals_ok:
                # Nothing valid in this file; skip
                continue

            vf = _build_select_eq(locals_ok)
            out_pat = tmpdir / f"buf{fi}_%06d.png"

            cmd = [
                "ffmpeg","-y","-hide_banner","-loglevel","error",
                "-i", str(src),
                "-vf", vf,
                "-vsync","vfr",
                str(out_pat)
            ]
            cp = subprocess.run(cmd, capture_output=True, text=True)
            if cp.returncode != 0:
                # Show helpful context
                raise RuntimeError(
                    f"ffmpeg extract failed for {src}\n"
                    f"vf={vf}\n"
                    f"stderr:\n{cp.stderr}"
                )

            # Map emitted images back to requested locals (emitted in ascending n)
            emitted = sorted(out_pat.parent.glob(f"buf{fi}_*.png"))
            if len(emitted) != len(locals_ok):
                # Not fatal; keep best-effort mapping, but warn in exception if zero
                if len(emitted) == 0:
                    raise RuntimeError(f"No frames emitted for {src} with vf={vf}")
                # Partial emission: zip shortest
            for loc, png in zip(locals_ok, emitted):
                extracted[(fi, int(loc))] = png

        # 2) Assemble in requested order (skipping any that failed extraction)
        seq_dir = tmpdir / "seq"
        seq_dir.mkdir(exist_ok=True)
        seq_files = []
        for i, row in S.iterrows():
            fi = int(row["file_idx"]); loc = int(row["local"])
            key = (fi, loc)
            if key not in extracted:
                continue
            src_png = extracted[key]
            dst = seq_dir / f"f_{len(seq_files)+1:06d}.png"
            try:
                os.link(src_png, dst)
            except Exception:
                shutil.copy2(src_png, dst)
            seq_files.append(dst)

        if not seq_files:
            raise RuntimeError("No frames extracted — check indices vs nb_frames and codec support.")

        # 3) Single encode pass
        cmd = [
            "ffmpeg","-y","-hide_banner","-loglevel","error",
            "-framerate", str(fps_use),
            "-i", str(seq_dir / "f_%06d.png"),
            "-c:v","libx264","-preset", preset, "-pix_fmt","yuv420p",
            str(out_path)
        ]
        cp = subprocess.run(cmd, capture_output=True, text=True)
        if cp.returncode != 0:
            raise RuntimeError(f"ffmpeg encode failed:\n{cp.stderr}")

        return {
            "requested": len(S),
            "rendered_frames": len(seq_files),
            "fps": fps_use,
            "preset": preset,
            "out": str(out_path),
        }
    except Exception:
        # keep tmpdir for inspection on error
        raise



In [12]:
date_folder_manual = "251005"
csv_path = "/data/big_rim/rsync_dcc_sum/Oct3V1mini_sorted/20240919-V1-R1/customEntValHere/2024_12_18/11_33_01/My_V4_Miniscope/timeStamps.csv"
video_dir = "/data/big_rim/rsync_dcc_sum/Oct3V1mini_sorted/20240919-V1-R1/customEntValHere/2024_12_18/11_33_01/My_V4_Miniscope/"
out_path  = f"/data/big_rim/rsync_dcc_sum/Oct3V1/2024_12_18/20240919v1l5r1mini_p20240717PMC_social_test_11_30/MIR_Aligned/vis/{date_folder_manual}/selected_frames.avi"

timestamps_ms = frames.index[mask] #[-5, 35, 64, 97]  # your exact list (ms) 

sel = map_exact_timestamps_to_frames_with_buffer(csv_path, timestamps_ms)

# info = write_frames_video_by_fileidx_ffmpeg_fast(
#     video_dir=video_dir,
#     selections_df=sel,       # needs columns: ['timestamp_ms','frame_number', ...]
#     out_path=out_path,
#     fps=None,                # use metaData.json frameRate
#     preset="ultrafast"
# )

# print(info)
# assert info["rendered_frames"] == len(sel)

# info

### colorful overlay tries

In [21]:
# grabbed from /home/lq53/mir_repos/BBOP/random_tests/25Sep_posters/250929_get_rawC_func_try.ipynb
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib import patheffects as pe
import scipy.sparse as sp

# ---------- robust A-handling ----------
def _as_array(A):
    """xarray/sparse -> plain squeezed ndarray."""
    if hasattr(A, "values"):
        A = A.values
    if sp.issparse(A):
        A = A.toarray()
    A = np.asarray(A)
    return np.squeeze(A)

def _make_mask_getter(A, H, W):
    """
    Return (n_rois, mask_of) where mask_of(i) -> (H,W) bool.
    Supports:
      (n, H*W), (H*W, n), (H, W, n), (n, H, W)
    """
    A = _as_array(A)
    if A.ndim == 2:
        n0, n1 = A.shape
        if n1 == H * W:           # (n, H*W)
            n = n0
            def mask_of(i): return (A[i] > 0).reshape(H, W)
            return n, mask_of
        if n0 == H * W:           # (H*W, n)
            n = n1
            def mask_of(i): return (A[:, i] > 0).reshape(H, W)
            return n, mask_of
        raise ValueError(f"A shape {A.shape} doesn't match H*W={H*W}.")
    elif A.ndim == 3:
        s0, s1, s2 = A.shape
        if (s0, s1) == (H, W):    # (H, W, n)
            n = s2
            def mask_of(i): return A[:, :, i] > 0
            return n, mask_of
        if (s1, s2) == (H, W):    # (n, H, W)
            n = s0
            def mask_of(i): return A[i, :, :] > 0
            return n, mask_of
        raise ValueError(f"A shape {A.shape} doesn't match (H,W,_) or (_,H,W).")
    else:
        raise ValueError(f"A ndim={A.ndim} not supported. Got shape {A.shape}.")

def _centroid(mask_bool):
    ys, xs = np.nonzero(mask_bool)
    if xs.size == 0: return None
    return float(xs.mean()), float(ys.mean())

# ---------- core painter ----------
def draw_roi_overlay_on_ax(
    ax,
    max_proj,
    A,
    selected,
    show_ids=True,
    lw_all=0.6,
    lw_sel=1.8,
    all_color="#bbbbbb",
    all_alpha=0.8,
):
    """
    Draw all ROI edges (light gray) and highlight `selected` in color.
    Returns {roi_idx: rgba_color} for selected.
    """
    # accept RGB or grayscale max_proj
    if max_proj.ndim == 2:
        H, W = max_proj.shape
    elif max_proj.ndim == 3:
        H, W = max_proj.shape[:2]
    else:
        raise ValueError(f"max_proj shape {max_proj.shape} unsupported.")

    n, mask_of = _make_mask_getter(A, H, W)

    selected = [int(i) for i in dict.fromkeys(selected) if 0 <= int(i) < n]
    # palette = [cm.get_cmap("tab20")(i % 20) for i in range(len(selected))]
    palette = [mpl_cmaps["tab20"](i % 20) for i in range(len(selected))]
    color_map = {roi: palette[i] for i, roi in enumerate(selected)}

    # background
    if max_proj.ndim == 2:
        ax.imshow(max_proj, cmap="gray", interpolation="nearest")
    else:
        ax.imshow(max_proj, interpolation="nearest")

    # all ROI edges (light gray)
    for r in range(n):
        m = mask_of(r)
        if m.any():
            ax.contour(m.astype(float), levels=[0.5], colors=[all_color],
                      linewidths=lw_all, alpha=all_alpha, zorder=2)

    # selected overlays (colored + optional ids)
    for roi in selected:
        m = mask_of(roi)
        if not m.any(): continue
        c = color_map[roi]
        ax.contour(m.astype(float), levels=[0.5], colors=[c],
                   linewidths=lw_sel, alpha=1.0, zorder=3)
        if show_ids:
            cen = _centroid(m)
            if cen is not None:
                txt = ax.text(cen[0], cen[1], str(roi), color="white", fontsize=10,
                              ha="center", va="center", zorder=4)
                txt.set_path_effects([pe.Stroke(linewidth=2.5, foreground="black"), pe.Normal()])

    ax.axis("off")
    return color_map

# ---------- single-image overlay ----------
def roi_overlay_colored(
    data,
    max_proj,
    selected,
    show_ids=True,
    figsize=(8, 8),
    **kwargs
):
    """
    Poster-ready overlay. All ROIs in light gray; selected highlighted in color.
    """
    A = data['A']
    fig, ax = plt.subplots(figsize=figsize)
    _ = draw_roi_overlay_on_ax(ax, max_proj, A, selected, show_ids=show_ids, **kwargs)
    ax.set_title("ROI overlay (colored = selected)" + (" + ids" if show_ids else ""))
    return fig, ax

# ---------- side-by-side overlay + dF/F ----------
def overlay_and_dff(
    data,
    max_proj,
    dff,        # (n_rois, T)
    ts,         # (T,)
    selected,
    show_ids=True,
    shift=6.0,
    figsize=(14, 8),
    **kwargs
):
    """
    Left: overlay as above. Right: stacked ΔF/F traces with matching colors.
    """
    A = data['A']
    fig = plt.figure(figsize=figsize)
    gs = fig.add_gridspec(1, 2, width_ratios=[1.0, 1.2], wspace=0.05)
    ax_img = fig.add_subplot(gs[0, 0])
    ax_plot = fig.add_subplot(gs[0, 1])

    color_map = draw_roi_overlay_on_ax(ax_img, max_proj, A, selected, show_ids=show_ids, **kwargs)

    selected = [int(i) for i in dict.fromkeys(selected)]
    for i, roi in enumerate(selected):
        c = color_map.get(roi, (0, 0, 0, 1))
        y = dff[roi]
        ax_plot.plot(ts, y + i * shift, lw=0.9, color=c)
        if show_ids:
            ax_plot.text(ts[0], (y + i * shift)[0], f"{roi}", fontsize=9, color=c, va="bottom")

    ax_plot.set_xlabel("Time (ms)")
    ax_plot.set_yticks([])
    ax_plot.set_title("ΔF/F (colors = ROI edges)" + (" + ids" if show_ids else ""))
    return fig, (ax_img, ax_plot)


In [27]:
# What's your current function called?
timeline_test = build_timeline_with_csv(sel, csv_path=None, video_dir=video_dir, merged=merged)
print(timeline_test[["timestamp_ms", "frame_number", "t_ix"]].head(10))
print(f"t_ix dtype: {timeline_test['t_ix'].dtype}")

   timestamp_ms  frame_number  t_ix
0          1910            57    57
1          1945            58    58
2          1978            59    59
3          2011            60    60
4          2044            61    61
5          2079            62    62
6          2113            63    63
7          2145            64    64
8          2182            65    65
9          2216            66    66
t_ix dtype: int64


In [29]:
def build_timeline_with_csv(sel: pd.DataFrame, csv_path: str, video_dir: str, merged: pd.DataFrame) -> pd.DataFrame:
    timeline = sel.copy()
    
    # Merged lookup: timestamp_ms → t_ix (row position in merged)
    merged_ts_to_idx = {int(ts): i for i, ts in enumerate(merged.index)}
    
    # Debug: check a few specific lookups
    print("\nDebug lookups:")
    for ts in timeline["timestamp_ms"].head(5):
        result = merged_ts_to_idx.get(int(ts), "NOT FOUND")
        print(f"  timestamp {ts} -> t_ix = {result}")
    
    timeline["t_ix"] = timeline["timestamp_ms"].map(merged_ts_to_idx)
    
    print(f"\nAfter mapping:")
    print(f"  t_ix NaN count: {timeline['t_ix'].isna().sum()}")
    print(f"  t_ix non-NaN count: {timeline['t_ix'].notna().sum()}")
    print(f"  First 5 t_ix values: {timeline['t_ix'].head().tolist()}")
    
    # Keep only rows that exist in merged
    before_drop = len(timeline)
    timeline = timeline.dropna(subset=["t_ix"])
    after_drop = len(timeline)
    print(f"  Dropped {before_drop - after_drop} rows with NaN t_ix")
    
    timeline["t_ix"] = timeline["t_ix"].astype(int)
    
    # Compute file_idx and local for video access
    fpf = int(load_miniscope_meta(video_dir)["framesPerFile"])
    timeline["file_idx"] = (timeline["frame_number"] // fpf).astype(int)
    timeline["local"] = (timeline["frame_number"] % fpf).astype(int)
    
    timeline = timeline.sort_values("timestamp_ms").reset_index(drop=True)
    
    return timeline

In [36]:
import re, numpy as np, pandas as pd
import xarray as xr, numpy as np
import subprocess, io
from pathlib import Path
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
# NEW: stable cmap (Matplotlib 3.7+)
from matplotlib import colormaps as mpl_cmaps

def build_dff_and_ts_from_merged(merged: pd.DataFrame, roi_pat=r"^dF_F_roi(\d+)$", time_col="timestamp_ms_mini"):
    def _to_ms(arr):
        a = np.asarray(arr)
        if np.issubdtype(a.dtype, np.datetime64):
            return (a.astype("datetime64[ns]").astype(np.int64) / 1_000_000.0).astype(np.float64)
        return a.astype(np.float64)

    if time_col in merged.columns:
        ts = _to_ms(merged[time_col].to_numpy())
    elif (merged.index.name and (merged.index.name == time_col or "ms" in str(merged.index.name).lower())):
        ts = _to_ms(merged.index.to_numpy())
    else:
        ms_like = [c for c in merged.columns if re.search(r"(timestamp.*ms|_ms)$", str(c), flags=re.I)]
        ts = _to_ms(merged[ms_like[0]].to_numpy()) if ms_like else _to_ms(merged.index.to_numpy())

    roi_cols, roi_ids = [], []
    for c in merged.columns:
        m = re.match(roi_pat, str(c))
        if m:
            roi_cols.append(c); roi_ids.append(int(m.group(1)))
    if not roi_cols:
        raise ValueError("No columns matching ^dF_F_roi(\\d+)$ in merged.")

    order = np.argsort(roi_ids)
    roi_cols_sorted = [roi_cols[i] for i in order]
    roi_ids_sorted  = [roi_ids[i] for i in order]
    dF_F = merged[roi_cols_sorted].to_numpy(dtype=np.float32).T  # (n_rois, T)
    roi_to_row = {rid: i for i, rid in enumerate(roi_ids_sorted)}
    return dF_F, ts, roi_ids_sorted, roi_to_row


def build_timeline_with_csv(sel: pd.DataFrame, csv_path: str, video_dir: str, merged: pd.DataFrame) -> pd.DataFrame:
    """
    Build complete timeline for rendering.
    sel already has frame_number, buffer_idx from CSV.
    Just need to add t_ix from merged.
    """
    timeline = sel.copy()
    
    # Merged lookup: timestamp_ms → t_ix (row position in merged)
    merged_ts_to_idx = {int(ts): i for i, ts in enumerate(merged.index)}
    timeline["t_ix"] = timeline["timestamp_ms"].map(merged_ts_to_idx)
    
    # Keep only rows that exist in merged
    timeline = timeline.dropna(subset=["t_ix"])
    timeline["t_ix"] = timeline["t_ix"].astype(int)
    
    # Compute file_idx and local for video access
    fpf = int(load_miniscope_meta(video_dir)["framesPerFile"])
    timeline["file_idx"] = (timeline["frame_number"] // fpf).astype(int)
    timeline["local"] = (timeline["frame_number"] % fpf).astype(int)
    
    # Sort by timestamp
    timeline = timeline.sort_values("timestamp_ms").reset_index(drop=True)
    
    return timeline


def load_A_from_nc(nc_path: str):
    ds = xr.open_dataset(nc_path, engine="netcdf4")
    try:
        if "A" not in ds:
            raise KeyError("Variable 'A' not found in dataset.")
        return np.array(ds["A"].data, dtype=np.float32)
    finally:
        ds.close()


def ffmpeg_grab_png_frame(video_file: Path, local_index: int) -> np.ndarray:
    """
    Returns HxWxC uint8 array for frame n=local_index from video_file.
    No disk IO. Uses image2pipe PNG over stdout.
    """
    cmd = [
        "ffmpeg","-hide_banner","-loglevel","error",
        "-i", str(video_file),
        "-vf", f"select='eq(n,{int(local_index)})'",
        "-vframes","1",
        "-f","image2pipe","-vcodec","png","-"
    ]
    cp = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    if cp.returncode != 0 or len(cp.stdout) == 0:
        raise RuntimeError(f"ffmpeg extract failed for {video_file} n={local_index}:\n{cp.stderr.decode('utf-8','ignore')}")
    img = Image.open(io.BytesIO(cp.stdout)).convert("RGB")
    return np.asarray(img)


def open_ffmpeg_encoder(out_path: str, fps: float = 30.0, preset: str = "ultrafast"):
    cmd = [
        "ffmpeg","-y","-hide_banner","-loglevel","error",
        "-f","image2pipe","-vcodec","png","-r", f"{fps}",  # incoming stream
        "-i","-",
        "-c:v","libx264","-preset", preset, "-pix_fmt","yuv420p",
        out_path
    ]
    return subprocess.Popen(cmd, stdin=subprocess.PIPE)

def compose_frame_to_png_bytes(
    frame_img, A, selected_neurons, roi_to_row, color_map,
    dF_F, ts, t_ix: int, shift: float = 6.0, figsize=(14, 8), 
    show_ids: bool = False, xlim=None, dpi=150,
    is_new_episode: bool = False
) -> bytes:
    fig = plt.figure(figsize=figsize)
    gs  = fig.add_gridspec(1, 2, width_ratios=[1.0, 1.35])
    axL = fig.add_subplot(gs[0, 0])
    axR = fig.add_subplot(gs[0, 1])

    # LEFT: colorful overlays
    draw_roi_overlay_on_ax(
        ax=axL, max_proj=frame_img, A=A, selected=selected_neurons,
        show_ids=show_ids, lw_all=0.6, lw_sel=1.8, all_color="#bbbbbb", all_alpha=0.8
    )

    # RIGHT: stacked ΔF/F
    axR.set_yticks([]); axR.set_xlabel("Time (ms)")
    if xlim is None: axR.set_xlim(float(ts[0]), float(ts[-1]))
    else:            axR.set_xlim(*xlim)
    
    # Episode separator
    if is_new_episode:
        axR.axvline(float(ts[t_ix]), lw=3.0, color="red", alpha=0.7, linestyle="--")
    else:
        if 0 <= t_ix < len(ts): 
            axR.axvline(float(ts[t_ix]), lw=1.0, color="k", alpha=0.5)

    for i, rid in enumerate(selected_neurons):
        row = roi_to_row.get(int(rid))
        if row is None: continue
        yseg = dF_F[row, : t_ix + 1]
        axR.plot(ts[: t_ix + 1], yseg + i * shift, lw=0.9, color=color_map[int(rid)])

    fig.subplots_adjust(left=0.05, right=0.98, top=0.96, bottom=0.10, wspace=0.04)

    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=dpi)
    plt.close(fig)
    return buf.getvalue()

def make_roi_color_map(selected_neurons):
    cmap = mpl_cmaps.get_cmap("tab20")   # or: mpl_cmaps["tab20"]
    return {int(r): cmap(i % 20) for i, r in enumerate(selected_neurons)}


def render_streaming_side_by_side(
    timeline: pd.DataFrame,
    video_dir: str,
    A,
    selected_neurons,
    roi_to_row,
    dF_F, ts,
    out_video_path: str,
    fps: float = None,
    show_ids: bool = False,
    shift: float = 6.0,
    dpi: int = 150,
    episode_padding_ms: float = 500.0,  # NEW: add padding around each episode
):
    video_dir = Path(video_dir)
    meta = load_miniscope_meta(video_dir)
    fps_use = float(meta["frameRate"] if fps is None else fps)

    color_map = make_roi_color_map(selected_neurons)
    
    # Detect episode boundaries
    timeline = timeline.copy()
    timeline["is_new_episode"] = False
    timeline.loc[0, "is_new_episode"] = True
    
    episode_id = 0
    timeline["episode_id"] = 0
    
    for i in range(1, len(timeline)):
        if timeline.loc[i, "t_ix"] - timeline.loc[i-1, "t_ix"] > 1:
            episode_id += 1
            timeline.loc[i, "is_new_episode"] = True
        timeline.loc[i, "episode_id"] = episode_id

    enc = open_ffmpeg_encoder(out_video_path, fps=fps_use, preset="ultrafast")
    try:
        for _, r in timeline.iterrows():
            fi  = int(r["file_idx"])
            loc = int(r["local"])
            src = video_dir / f"{fi}.avi"
            
            frame_img = ffmpeg_grab_png_frame(src, loc)
            
            # NEW: Compute xlim for current episode only
            current_episode = timeline[timeline["episode_id"] == r["episode_id"]]
            episode_t_ix = current_episode["t_ix"].values
            episode_ts = ts[episode_t_ix]
            
            xlim = (
                float(episode_ts.min() - episode_padding_ms),
                float(episode_ts.max() + episode_padding_ms)
            )
            
            png_bytes = compose_frame_to_png_bytes(
                frame_img=frame_img, A=A, selected_neurons=selected_neurons, roi_to_row=roi_to_row,
                color_map=color_map, dF_F=dF_F, ts=ts, t_ix=int(r["t_ix"]),
                shift=shift, show_ids=show_ids, xlim=xlim, dpi=dpi,
                is_new_episode=bool(r["is_new_episode"])
            )

            enc.stdin.write(png_bytes)

    finally:
        enc.stdin.close()
        enc.wait()


In [37]:
selected_neurons = [10, 21,  3, 36, 11, 33, 25,  5, 32,  9, 14, 23, 35, 37,  2,  0, 17, 20, 34, 12]
csv_path = "/data/big_rim/rsync_dcc_sum/Oct3V1mini_sorted/20240919-V1-R1/customEntValHere/2024_12_18/11_33_01/My_V4_Miniscope/timeStamps.csv"
# minian_nc_path = "/data/big_rim/rsync_dcc_sum/Oct3V1mini_sorted/20240919-V1-R1/customEntValHere/2024_12_18/11_33_01/My_V4_Miniscope/minian_dataset_wnd1500_stp700_max25_diff3.5_pnr1.1.nc"
minian_nc_path = "/data/big_rim/rsync_dcc_sum/Oct3V1mini_sorted/20240919-V1-R1/customEntValHere/2024_12_18/11_33_01/My_V4_Miniscope/minian_dataset_wnd1000_stp700_max25_diff3.5_pnrauto.nc"

In [17]:
print("sel columns:", sel.columns.tolist())
print("sel timestamp_ms sample:", sel["timestamp_ms"].head())
print("sel timestamp_ms type:", sel["timestamp_ms"].dtype)

sel columns: ['timestamp_ms', 'frame_number', 'buffer_idx']
sel timestamp_ms sample: 0    1910
1    1945
2    1978
3    2011
4    2044
Name: timestamp_ms, dtype: int64
sel timestamp_ms type: int64


In [ ]:
# You already have:
# - merged = merge_pred_with_miniscope(...)
# - sel with at least ['timestamp_ms','frame_number'] (and optionally file_idx/local)
# - video_dir = ".../My_V4_Miniscope"   (same folder as your split 0.avi, 1.avi, ...)
# - selected_neurons = [...]

# Build signals and sync
# dF_F, ts, roi_ids, roi_to_row = build_dff_and_ts_from_merged(merged)
# timeline = add_t_ix(sel, ts)

# # If sel doesn’t have file_idx/local, compute them once (no extra IO)
# if "file_idx" not in timeline or "local" not in timeline:
#     fpf = int(load_miniscope_meta(video_dir)["framesPerFile"])
#     timeline["file_idx"] = (timeline["frame_number"] // fpf).astype(int)
#     timeline["local"]    = (timeline["frame_number"] %  fpf).astype(int)

dF_F, ts, roi_ids, roi_to_row = build_dff_and_ts_from_merged(merged)
timeline = build_timeline_with_csv(sel, csv_path=None, video_dir=video_dir, merged=merged)

# Load A from the specific .nc you use
A_np = load_A_from_nc(minian_nc_path)

# Stream: no intermediate PNGs, overlays appear in the final video
out_video = "./tmp/new_3hift_side_by_side_stream_wnd1000_stp700_max25_diff3.5_pnrauto.mp4"
render_streaming_side_by_side(
    timeline=timeline,
    video_dir=video_dir,
    A=A_np,
    selected_neurons=selected_neurons,
    roi_to_row=roi_to_row,
    dF_F=dF_F, ts=ts,
    out_video_path=out_video,
    fps=None,            # use metaData.json frameRate
    show_ids=False,
    shift=3.0, # add more with less timepoints, full with 6 looks ok.
    dpi=130,             # tweak for poster crispness (130–150 is typical)
)
print("done:", out_video)


In [31]:
print("\nChecking timeline continuity:")
print(f"t_ix values: {timeline['t_ix'].values}")
print(f"Are they consecutive? {np.all(np.diff(timeline['t_ix']) == 1)}")


Checking timeline continuity:
t_ix values: [  57   58   59   60   61   62   63   64   65   66   67   68   69   70
   71   72   73   74   75   76   77   78   79   80   81   82   83   84
   85   86   87   88   89   90   91   92   93   94   95   96   97   98
   99  100  101  102  103  104 5680 5681 5682 5683 5684 5685 5686 5687
 5688 5689 5690 5691 5692 5693 5694 5695 5696 5697 5698 5699 5700 5701
 5702 5703 5704 5705 5706 5707 5708 5709 5710 5711 5712 5713 5714 5715
 5716 5717 5718 5719 5720 5721 5722 5723 5724 5725 5726 5727 5728 5729
 5730 5731 5732 5733 5734 5735 5736 5737 5738 5739 5740 5741 5742 5743
 5744 5745 5746 5747 5748 5749 5750 5751 5752 5753 8058 8059 8060 8061
 8062 8063 8064 8065 8066 8067 8068 8069 8070 8071 8072 8073 8074 8075
 8076 8077 8078 8079 8080 8081 8082 8083 8084 8085 8086 8087 8088 8089
 8090 8091 8092 8093 8094 8095 8096 8097 8098 8099 8100 8101 8102 8103
 8104 8105 8106 8107 8108 8109 8110 8111 8112 8113 8114 8115 8116 8117
 8118 8119 8120 8121 8122 8123 81